python -m venv venv
venv\Scripts\activate
pip install numpy matplotlib tensorflow scikit-learn
pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
pip install notebook
python -m notebook

Implement Artificial Neural Network training process in Python by using Forward Propagation,
Back Propagation.

In [1]:
import numpy as np
import pandas as pd

In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        self.W1 = np.random.randn(input_size, hidden_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size)
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def sigmoid_derivative(self, x):
        return x * (1 - x)

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)

        return self.a2

    def backward(self, X, y, output, learning_rate):
        output_error = y - output
        output_delta = output_error * self.sigmoid_derivative(output)

        hidden_error = np.dot(output_delta, self.W2.T)
        hidden_delta = hidden_error * self.sigmoid_derivative(self.a1)

        self.W2 += np.dot(self.a1.T, output_delta) * learning_rate
        self.b2 += np.sum(output_delta, axis=0, keepdims=True) * learning_rate

        self.W1 += np.dot(X.T, hidden_delta) * learning_rate
        self.b1 += np.sum(hidden_delta, axis=0, keepdims=True) * learning_rate

    def train(self, X, y, epochs, learning_rate):
        for epoch in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output, learning_rate)

            if epoch % 1000 == 0:
                loss = np.mean(np.square(y - output))
                print(f"Epoch {epoch}, Loss: {loss}")

In [ ]:
np.random.seed(42)

n_samples = 200
X = np.random.uniform(-2, 2, (n_samples, 2))

# Inside circle (radius = 1.5)
y = (X[:, 0]**2 + X[:, 1]**2 < 1.5**2).astype(int).reshape(-1, 1)

X[:5], y[:5]

(array([[-0.50183952,  1.80285723],
        [ 0.92797577,  0.39463394],
        [-1.37592544, -1.37602192],
        [-1.76766555,  1.46470458],
        [ 0.40446005,  0.83229031]]),
 array([[0],
        [1],
        [0],
        [0],
        [1]]))

In [ ]:
train_size = 150

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

In [10]:
nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1)

print("=== BEFORE TRAINING ===")
print(nn.forward(X)[:10])

=== BEFORE TRAINING ===
[[0.47356815]
 [0.47954401]
 [0.42409094]
 [0.46066457]
 [0.48148719]
 [0.46246459]
 [0.46176556]
 [0.42559743]
 [0.45004604]
 [0.44133434]]


In [11]:
nn.train(X_train, y_train, epochs=10000, learning_rate=0.1)

Epoch 0, Loss: 0.2456742081427712
Epoch 1000, Loss: 0.015288965365632052
Epoch 2000, Loss: 0.010377123456803855
Epoch 3000, Loss: 0.00824625794474096
Epoch 4000, Loss: 0.006985416459913498
Epoch 5000, Loss: 0.00612840336107107
Epoch 6000, Loss: 0.0054945931185437875
Epoch 7000, Loss: 0.004998843679889102
Epoch 8000, Loss: 0.004595376795135334
Epoch 9000, Loss: 0.004257207470660605


In [7]:
print("\n=== AFTER TRAINING ===")
print(nn.forward(X)[:5])


=== AFTER TRAINING ===
[[9.04252002e-03]
 [9.99979341e-01]
 [5.54883724e-04]
 [5.20446493e-06]
 [9.99999824e-01]]


In [8]:
predictions = nn.forward(X_test)
binary_preds = (predictions > 0.5).astype(int)

inside_circle = np.where(binary_preds == 1, "Yes", "No")
accuracy = np.mean(binary_preds == y_test)

print(f"Test Accuracy: {accuracy * 100:.2f}%")

Test Accuracy: 88.00%


In [9]:
preds_df = pd.DataFrame({
    "Input (X1)": X_test[:, 0],
    "Input (X2)": X_test[:, 1],
    "Predicted": binary_preds[:, 0],
    "Actual": y_test[:, 0],
    "Inside Circle": inside_circle[:, 0]
})

preds_df.head()

,Input (X1),Input (X2),Predicted,Actual,Inside Circle
0,-1.793273,0.125419,0,0,No
1,0.162540,0.549720,1,1,Yes
2,0.904365,1.903408,0,0,No
3,0.065201,-0.708174,1,1,Yes
4,1.180745,-0.916671,1,1,Yes
